# 1. Workbench Introduction
이 notebook은 Kaggle T4에서 repository-owned YOLO experiment를 검증·실행·시각화하는 thin interface다. C4-2A/C4-2B 선택에는 train/validation만 사용하며 derived test는 봉인한다.

## 2. Mode / Experiment Controls
Research는 비공식 임시 override, Official은 committed config만 사용한다.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

MODE = "research"  # 'research' | 'official'
RUN_RESEARCH_SMOKE = False
RUN_OFFICIAL_TRAINING = False
RESEARCH_OVERRIDES = {"imgsz": 768, "batch": 8, "epochs": 3, "patience": 2}
OFFICIAL_OVERRIDES = {}  # 반드시 비어 있어야 한다.
OFFICIAL_EXPERIMENT_ID = os.environ.get(
    "SMARTFACTORY_YOLO_EXPERIMENT_ID", "c4_2a_yolo11n_seg_imgsz1024_seed42"
)
REPOSITORY_ROOT = Path(
    os.environ.get("SMARTFACTORY_REPOSITORY_ROOT", "/kaggle/working/smart-factory-ai-platform")
)
DATASET_ROOT = Path(
    os.environ.get(
        "SMARTFACTORY_YOLO_DATASET_ROOT",
        str(
            REPOSITORY_ROOT
            / "data/processed/supervised_derived/mvtec_ad/metal_nut/yolo_segmentation/v1"
        ),
    )
)
BASELINE_ARTIFACT_DIR = Path(
    os.environ.get(
        "SMARTFACTORY_YOLO_BASELINE_ARTIFACT_DIR",
        str(
            REPOSITORY_ROOT
            / "artifacts/runtime/yolo_segmentation/smartfactory_yolo11n_seg_metal_nut_seed42_t4"
        ),
    )
)
OFFICIAL_CONFIG_PATH = (
    REPOSITORY_ROOT / f"configs/experiments/yolo_segmentation/{OFFICIAL_EXPERIMENT_ID}.yaml"
)
WORKBENCH_OUTPUT_ROOT = (
    REPOSITORY_ROOT / "outputs/workbench/yolo_segmentation" / OFFICIAL_EXPERIMENT_ID
)

## 3. Environment Verification
동일 `pyproject.toml`과 `uv.lock`을 sync한다. Notebook kernel은 display/controller이며, model-affecting computation은 `uv run --locked python` provenance로 검증한다.

In [ ]:
subprocess.run(["uv", "sync", "--locked"], cwd=REPOSITORY_ROOT, check=True)
site_packages = next((REPOSITORY_ROOT / ".venv/lib").glob("python*/site-packages"))
sys.path[:0] = [str(REPOSITORY_ROOT), str(site_packages)]
from IPython.display import Image as NotebookImage  # noqa: E402
from IPython.display import display  # noqa: E402

from ml.experiments.yolo_workbench_runtime import inspect_locked_environment  # noqa: E402

locked_environment = inspect_locked_environment(REPOSITORY_ROOT, require_cuda=MODE == "official")
print({"notebook_controller_python": sys.executable, "role": "DISPLAY_CONTROLLER_ONLY"})
display(locked_environment.to_json_dict())

## 4. Repository / Git Verification
HEAD와 dirty state는 official provenance에 기록한다.

In [ ]:
from ml.evaluation.final_benchmark import resolve_repository_provenance

repository_provenance = resolve_repository_provenance(REPOSITORY_ROOT)
repository_provenance.to_json_dict()

## 5. Dataset / Manifest Verification
Manifest 전체 identity를 검증하되 workbench row는 train/validation만 materialize한다. TEST SPLIT = SEALED / NOT USED.

In [ ]:
from ml.experiments.yolo_segmentation import load_yolo_experiment_config
from ml.training.yolo_segmentation import (
    load_yolo_segmentation_config,
    validate_experiment_dataset,
)

experiment = load_yolo_experiment_config(OFFICIAL_CONFIG_PATH)
baseline = load_yolo_segmentation_config(experiment.baseline_config_path)
records = list(validate_experiment_dataset(DATASET_ROOT, baseline.dataset_contract))
print(
    {
        "dataset": baseline.dataset_contract.category,
        "manifest_sha256": baseline.dataset_contract.manifest_sha256,
        "classes": baseline.dataset_contract.classes,
        "allowed_rows": len(records),
        "test_split": "SEALED_NOT_USED",
    }
)

## 6. Experiment Configuration Summary
Official mode는 config의 단일 controlled intervention만 적용한다. Research override는 committed config를 mutate하지 않는다.

In [ ]:
from dataclasses import asdict

from ml.experiments.yolo_workbench import build_research_config, validate_workbench_controls

validate_workbench_controls(
    MODE, overrides=OFFICIAL_OVERRIDES if MODE == "official" else RESEARCH_OVERRIDES
)
active_config = (
    experiment.training_config(baseline)
    if MODE == "official"
    else build_research_config(
        baseline, overrides=RESEARCH_OVERRIDES, output_root=WORKBENCH_OUTPUT_ROOT / "research"
    )
)
print(
    {
        "mode": MODE,
        "official_experiment_id": experiment.experiment_id,
        "controlled_change": asdict(experiment.controlled_change),
        "training": asdict(active_config.training),
        "label": "OFFICIAL" if MODE == "official" else "NON_OFFICIAL_RESEARCH",
    }
)

## 7. Dataset EDA
Image/class/component/size 분포는 C4-1 mask-area-ratio boundary를 재사용하며 test를 읽지 않는다.

In [ ]:
from ml.experiments.yolo_workbench import (
    build_eda_summary,
    describe_workbench_samples,
    select_representative_samples,
    select_small_validation_sample,
    write_eda_summary,
)
from ml.experiments.yolo_workbench_visualization import render_eda_distribution

size_policy = experiment.validation_protocol.size_policy()
samples = describe_workbench_samples(
    records,
    dataset_root=DATASET_ROOT,
    classes=baseline.dataset_contract.classes,
    size_policy=size_policy,
)
eda = build_eda_summary(
    samples,
    manifest_sha256=baseline.dataset_contract.manifest_sha256,
    dataset_name=records[0].dataset_name,
    dataset_version=records[0].dataset_version,
)
write_eda_summary(eda, WORKBENCH_OUTPUT_ROOT / "eda_summary.json")
display(eda)
eda_figure = render_eda_distribution(
    eda, WORKBENCH_OUTPUT_ROOT / "visualizations/dataset/dataset_distribution.png"
)
display(NotebookImage(filename=str(eda_figure)))

## 8. Ground Truth Visualization
Stable SHA ranking으로 class/size/single·multi/good coverage를 선택하고 sample ID와 GT overlay를 표시한다.

In [ ]:
from ml.experiments.yolo_workbench_visualization import render_ground_truth_gallery

representatives = select_representative_samples(samples, seed=baseline.training.seed)
gt_gallery = render_ground_truth_gallery(
    samples=representatives,
    dataset_root=DATASET_ROOT,
    classes=baseline.dataset_contract.classes,
    output_path=WORKBENCH_OUTPUT_ROOT / "visualizations/dataset/ground_truth_gallery.png",
)
display(NotebookImage(filename=str(gt_gallery)))

## 9. Actual Training Augmentation Preview
Pinned Ultralytics `build_yolo_dataset`와 실제 `build_transforms` path를 `uv run --locked python` preview subprocess에서 호출한다. Notebook system package는 official preview evidence에 사용하지 않는다.

In [ ]:
from ml.experiments.yolo_workbench_runtime import run_locked_preview_subprocess

train_ids = [
    item.sample_id for item in representatives if item.split == "train" and not item.is_negative
][:3]
representation_sample_id = None
if experiment.intervention_type == "resolution":
    small_sample = select_small_validation_sample(samples, seed=baseline.training.seed)
    representation_sample_id = small_sample.sample_id
preview_artifacts = run_locked_preview_subprocess(
    mode=MODE,
    experiment_config_path=OFFICIAL_CONFIG_PATH,
    dataset_root=DATASET_ROOT,
    output_root=WORKBENCH_OUTPUT_ROOT,
    repository_root=REPOSITORY_ROOT,
    train_sample_ids=train_ids,
    representation_sample_id=representation_sample_id,
    research_overrides=RESEARCH_OVERRIDES if MODE == "research" else OFFICIAL_OVERRIDES,
)
augmentation_figure = preview_artifacts.augmentation_figure
display(NotebookImage(filename=str(augmentation_figure)))
display(preview_artifacts.metadata["augmentation"])

## 10. Controlled Intervention Preview
C4-2A는 640/1024 representation을, C4-2B는 train-only sampling exposure와 eligibility를 각각 검증한다.

In [ ]:
import json

from ml.experiments.yolo_workbench import build_sampling_workbench_summary
from ml.experiments.yolo_workbench_visualization import write_visualization_manifest

intervention_entries = []
if experiment.intervention_type == "resolution":
    representation_figure = preview_artifacts.representation_figure
    if representation_figure is None:
        raise RuntimeError("Locked resolution preview was not generated.")
    display(NotebookImage(filename=str(representation_figure)))
    intervention_entries.append(
        {
            "visualization_type": "imgsz_640_vs_1024",
            "source_split": "val",
            "selected_sample_ids": [small_sample.sample_id],
            "generated_path": str(representation_figure),
        }
    )
elif experiment.intervention_type == "train_sampling_multiplicity":
    sampling_summary = build_sampling_workbench_summary(
        experiment=experiment,
        baseline=baseline,
        dataset_root=DATASET_ROOT,
        records=records,
    )
    if sampling_summary is None:
        raise RuntimeError("C4-2B sampling summary was not generated.")
    summary_path = WORKBENCH_OUTPUT_ROOT / "train_view_preflight.json"
    summary_path.write_text(
        json.dumps(sampling_summary, indent=2, sort_keys=True) + "\n", encoding="utf-8"
    )
    display({key: value for key, value in sampling_summary.items() if key != "eligible_samples"})
    display(sampling_summary["eligible_samples"])
else:
    raise ValueError("Unsupported Workbench intervention type.")
write_visualization_manifest(
    output_path=WORKBENCH_OUTPUT_ROOT / "visualization_manifest.json",
    experiment_id=experiment.experiment_id,
    manifest_sha256=baseline.dataset_contract.manifest_sha256,
    repository=repository_provenance.to_json_dict(),
    entries=[
        {
            "visualization_type": "dataset_distribution",
            "source_split": "train_val",
            "selected_sample_ids": [],
            "generated_path": str(eda_figure),
        },
        {
            "visualization_type": "ground_truth_gallery",
            "source_split": "train_val",
            "selected_sample_ids": [item.sample_id for item in representatives],
            "generated_path": str(gt_gallery),
        },
        {
            "visualization_type": "actual_training_augmentation",
            "source_split": "train",
            "selected_sample_ids": train_ids,
            "generated_path": str(augmentation_figure),
        },
        *intervention_entries,
    ],
)

## 11. Training Preflight
Official 실행 직전에 config/Manifest/Baseline/Git/split과 locked `.venv` interpreter/framework/CUDA identity를 fail-fast한다.

In [ ]:
from ml.experiments.yolo_workbench import WorkbenchPaths, build_official_preflight

paths = WorkbenchPaths(REPOSITORY_ROOT, DATASET_ROOT, BASELINE_ARTIFACT_DIR, WORKBENCH_OUTPUT_ROOT)
if MODE == "official":
    official_preflight = build_official_preflight(
        experiment=experiment,
        baseline=baseline,
        paths=paths,
        requested_device="cuda",
        overrides=OFFICIAL_OVERRIDES,
        execution_environment=locked_environment,
        repository_provenance=repository_provenance,
    )
    print("OFFICIAL EXPERIMENT PREFLIGHT")
    display(official_preflight.to_json_dict())
else:
    print("NON-OFFICIAL / RESEARCH — official preflight skipped")

## 12. Training Execution
Notebook을 여는 것만으로 학습하지 않는다. Official은 boolean을 직접 `True`로 바꾼 경우에만 `uv run --locked python -m pipelines.run_yolo_segmentation_experiment` subprocess를 호출한다.

In [ ]:
from ml.experiments.yolo_workbench_runtime import run_official_training_subprocess
from pipelines.train_yolo_segmentation import train_yolo_segmentation

experiment_artifacts = None
if MODE == "official" and RUN_OFFICIAL_TRAINING:
    experiment_artifacts = run_official_training_subprocess(
        experiment_config_path=OFFICIAL_CONFIG_PATH,
        experiment_id=experiment.experiment_id,
        experiment_root=experiment.output.experiment_root,
        artifact_root=experiment.output.artifact_root,
        package_root=experiment.output.package_root,
        dataset_root=DATASET_ROOT,
        baseline_artifact_dir=BASELINE_ARTIFACT_DIR,
        repository_root=REPOSITORY_ROOT,
    )
elif MODE == "research" and RUN_RESEARCH_SMOKE:
    research_artifact = train_yolo_segmentation(
        config=active_config,
        dataset_root=DATASET_ROOT,
        artifact_id="research-yolo-segmentation-smoke",
        requested_device="cuda",
    )
    print("NON-OFFICIAL / RESEARCH", research_artifact)
else:
    print("SKIPPED: RUN_OFFICIAL_TRAINING=False / RUN_RESEARCH_SMOKE=False")

## 13. Epoch Progress / Timing
`epoch_metrics.jsonl`은 `on_train_epoch_start`부터 `on_fit_epoch_end`까지의 measured fit-epoch elapsed time을 기록한다. Training batch뿐 아니라 해당 epoch의 validation/metric/checkpoint 처리도 포함하며 end-to-end wall-clock과는 별도 boundary다.

In [ ]:
import json

from ml.experiments.yolo_workbench_visualization import render_epoch_curves

if experiment_artifacts:
    epoch_path = experiment_artifacts.experiment_dir / "epoch_metrics.jsonl"
    display(
        NotebookImage(
            filename=str(
                render_epoch_curves(
                    epoch_path,
                    experiment_artifacts.experiment_dir
                    / "visualizations/training/training_curves.notebook.png",
                )
            )
        )
    )
    display([json.loads(line) for line in epoch_path.read_text().splitlines()[-5:]])

## 14. GPU Telemetry Summary
Existing sampler evidence를 읽는다. PyTorch allocator peak와 device-wide `nvidia-smi` sampling을 동일 값으로 해석하지 않는다.

In [ ]:
from ml.experiments.yolo_workbench_visualization import render_gpu_telemetry

if experiment_artifacts:
    display(json.loads(experiment_artifacts.telemetry_path.read_text()))
    display(
        NotebookImage(
            filename=str(
                render_gpu_telemetry(
                    experiment_artifacts.telemetry_path,
                    experiment_artifacts.experiment_dir
                    / "visualizations/training/gpu_telemetry.notebook.png",
                )
            )
        )
    )

## 15. Validation Metrics
Ultralytics framework metric과 C4 diagnostic metric의 protocol label을 유지하며 existing JSON을 표시한다.

In [ ]:
if experiment_artifacts:
    display(
        json.loads((experiment_artifacts.experiment_dir / "validation_metrics.json").read_text())
    )

## 16. Validation Failure Analysis
C4-1 matching/taxonomy가 생성한 deterministic val-only galleries와 per-class/size/component/negative summary를 검토한다.

In [ ]:
if experiment_artifacts:
    failure_root = (
        experiment_artifacts.experiment_dir
        / "candidate_error_analysis/visualizations/validation_failures"
    )
    display(json.loads((failure_root / "visualization_manifest.json").read_text()))
    display(
        json.loads(
            (experiment_artifacts.experiment_dir / "error_analysis_summary.json").read_text()
        )["aggregate"]
    )

## 17. Baseline vs Candidate Comparison
동일 validation sample을 regression-first로 포함하고 small/multi/complete-miss hypothesis 사례를 함께 본다. 개선 사례만 cherry-pick하지 않는다.

In [ ]:
if experiment_artifacts:
    display(json.loads(experiment_artifacts.comparison_path.read_text()))
    display(
        NotebookImage(
            filename=str(
                experiment_artifacts.experiment_dir
                / "visualizations/baseline_vs_candidate/baseline_vs_candidate_gallery.png"
            )
        )
    )

## 18. Artifact / Evidence Review
Model/config/evidence package와 SHA sidecar를 existing runner output에서 검토한다. Notebook-only 값은 source of truth가 아니다.

In [ ]:
if experiment_artifacts:
    display(json.loads(experiment_artifacts.experiment_result_path.read_text()))
    display(json.loads(experiment_artifacts.package_metadata_path.read_text()))

## 19. Final Test Review — LOCKED FOR C4 EXPERIMENTS
**FINAL TEST REVIEW: LOCKED**

C4-2 experiment selection이 진행 중이다. Derived test row를 이 section에서 load하거나 inspect하지 않는다. C4-3에서 validation-only 최종 후보 선택, artifact v2 freeze 후 별도 test namespace로 한 번만 평가한다.

In [ ]:
print("FINAL TEST REVIEW: LOCKED — no test data loaded")

## 20. Export Summary
Official run이 끝난 경우 existing ZIP과 metadata를 Kaggle output에서 export한다. Raw dataset/cache/model mount를 Git에 추가하지 않는다.

In [ ]:
if experiment_artifacts:
    print(
        {
            "package": str(experiment_artifacts.package_path),
            "package_metadata": str(experiment_artifacts.package_metadata_path),
            "result": str(experiment_artifacts.experiment_result_path),
        }
    )
else:
    print("No official package: training was not executed.")